# 🚀 OCR in Python mit PyTesseract 

```{admonition} Hinweise zur Ausführung des Notebooks
:class: hinweis
Dieses Notebook kann auf unterschiedlichen Levels erarbeitet werden (siehe Abschnitt ["Technische Voraussetzungen"](../introduction/introduction_requirements)):

1. **Book-Only Mode:** Sie lesen das Notebook hier im "Jupyter Book", ohne den Code selbst auszuführen.
2. **Cloud Mode:** Klicken Sie **oben rechts in der Menüleiste** auf das Raketen-Symbol <span class="launch-colab-inline">🚀</span> und wählen Sie "Colab", um das Notebook auszuführen.
3. **Local Mode:** Klicken Sie **oben rechts in der Menüleiste** auf das Download-Symbol <span class="launch-ipynb-inline">↓</span> und wählen Sie ".ipynb", um das Notebook lokal auszuführen.
```

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
  
<b>Voraussetzungen zur Ausführung des Jupyter Notebooks</b>
<ol>
<li> Installieren der Bibliotheken </li>
<li>Laden der Daten (s.u.)</li>
<li>Pfad zu den Daten setzen</li>
</ol>
Zum Testen: Ausführen der Zelle „load libraries“ und der Sektion „Einlesen des Texts“. </br>
Alle Zellen, die mit 🚀 gekennzeichnet sind, werden nur bei der Ausführung des Notebooks in Colab / JupyterHub bzw. lokal ausgeführt. 
</details>

## OCR mit Python

In diesem Notebook werden wir pyTesseract ausführen, um maschinenlesbaren Text zu erzeugen aus:
* einem JPEG-Bild
* einem mehrseitigen PDF
* einem Korpus mehrseitiger PDFs

<!--In this notebook, we will run pyTesseract to produce machine readable text from:
* a JPEG image
* a multi-paged PDF
* a corpus of multi-page PDF-s-->

## Installationen und Importe <!--Importing tools-->

In [ ]:
# 🚀 Install libraries
import sys
if 'google.colab' in sys.modules:
    !sudo apt-get update
    !sudo apt-get install -y tesseract-ocr poppler-utils
    # Das Fraktur-Modell deu_latf (früher frk) ist nicht als apt-Paket verfügbar – Modelldatei direkt laden
    !sudo wget -q https://github.com/tesseract-ocr/tessdata/raw/main/deu_latf.traineddata -P $(find /usr/share/tesseract-ocr -name tessdata -type d | head -1)
!pip install pytesseract pillow requests
!pip install pdf2image
!pip install tqdm

In [ ]:
import pytesseract
from PIL import Image
from pathlib import Path
from pdf2image import convert_from_path
from tqdm import tqdm

In [ ]:
# Windows: pytesseract findet tesseract.exe nur, wenn der Installationspfad
# in der PATH-Variable eingetragen ist. Falls nicht, suchen wir an den
# üblichen Installationsorten und setzen den Pfad für diese Session.
import os
import shutil
import sys

if sys.platform == "win32" and shutil.which("tesseract") is None:
    search_dirs = [
        Path(os.environ.get("PROGRAMFILES", r"C:\Program Files")),
        Path(os.environ.get("PROGRAMFILES(X86)", r"C:\Program Files (x86)")),
        Path(os.environ.get("LOCALAPPDATA", "")) / "Programs",
    ]
    for search_dir in search_dirs:
        tesseract_exe = search_dir / "Tesseract-OCR" / "tesseract.exe"
        if tesseract_exe.exists():
            pytesseract.pytesseract.tesseract_cmd = str(tesseract_exe)
            print(f"Tesseract gefunden: {tesseract_exe}")
            break
    else:
        print(
            "Tesseract wurde nicht automatisch gefunden. Bitte den Pfad zu Ihrer "
            "Installation manuell setzen, z.B.:\n"
            r"pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'"
        )

Bevor wir Daten herunterladen, definieren wir eine kleine Hilfsfunktion `download_file`. Sie lädt eine Datei plattformunabhängig – also auch unter Windows – aus dem Internet in einen Zielordner herunter und ersetzt damit das Kommando `wget`, das nicht auf allen Systemen (z.B. Windows) nativ verfügbar ist.

In [ ]:
# helper: download a single file (cross-platform replacement for `! wget -P`)
import requests
from pathlib import Path

def download_file(url, target_dir):
    """Download the file at `url` into `target_dir`, keeping its original name."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / url.split("/")[-1]
    response = requests.get(url)
    response.raise_for_status()
    target_path.write_bytes(response.content)
    return target_path

## Verarbeitung eines Bildes <!--Processing one image-->

In [ ]:
if not Path("grippe.jpeg").exists():
    download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/assets/images/grippe.jpeg", ".")

<img src='https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/assets/images/grippe.jpeg' width="600">

So können wir **OCR auf dieses Bild** des Zeitungsartikels ('Die Grippe wütet weiter') durchführen: <!-- This is how we can **perform OCR on this image** of the ('*Die Grippe wütet weiter*') newspaper article: -->

In [ ]:
ocr_output = pytesseract.image_to_string(Image.open('grippe.jpeg'), lang='deu_latf') 
print(ocr_output)

## Verschiedene Typen von OCR-Fehlern
Betrachten wir dieses Beispiel, so fallen sofort zahlreiche Fehler auf. Bereits das allererste Zeichen des ersten Wortes ist falsch: „7ie“ statt „Die“. Dies ist ein sehr häufiger OCR-Fehlertyp, bei dem ein Zeichen mit einem anderen verwechselt wird — in diesem Fall liegt die Ursache vermutlich unter anderem in der unterschiedlichen Druckfarbintensität verschiedener Bereiche des Buchstabens „D“. Dasselbe ist bei beiden „t“-Buchstaben im Wort „wütet“ geschehen: Sie wurden fälschlicherweise als „f“ bzw. „l“ erkannt. Solche Fehler bezeichnen wir als Substitutionsfehler.

Manchmal werden Zeichen nicht durch andere ersetzt, sondern gar nicht erst erkannt. Dies ist beispielsweise bei den meisten Zeichen im Wort „Angestellte“ der Fall — es wurde lediglich als „An«“ ausgegeben. Derartige Fehler lassen sich als Auslassungsfehler (engl. omission errors) klassifizieren.

Darüber hinaus treten gelegentlich zusätzliche Zeichen auf, die im Original nicht vorhanden sind. Dieser Fehlertyp lässt sich allerdings nicht immer eindeutig von einer Substitution abgrenzen — etwa dann, wenn ein einzelnes Zeichen der Vorlage im OCR-Ergebnis zu zwei oder mehr Zeichen wird.

Schließlich gibt es Abweichungen zwischen dem gewünschten und dem tatsächlichen OCR-Ergebnis, die sich nicht als Fehler im engeren Sinne einordnen lassen, sondern vielmehr Unterschiede in den Normalisierungskonventionen darstellen. So finden wir in der Originalquelle das lange s (ſ, U+017F, LATIN SMALL LETTER LONG S) an Stellen, an denen die moderne Orthographie ein gewöhnliches s (U+0073) vorsieht. Technisch handelt es sich dabei um zwei verschiedene Unicode-Zeichen. Ob eine Normalisierung vorgenommen wird oder nicht, ist letztlich eine Frage der Konvention und des jeweiligen Erkenntnisinteresses. Viele OCR-Engines normalisieren solche historischen Zeichenvarianten stillschweigend, andere bewahren sie im Sinne der editionsphilologischen Treue. Tesseract nimmt hier keine Normalisierung vor.

### 🚀 Selbst ausprobieren

Nutzen Sie das interaktive Werkzeug unten, um die OCR-Fehler direkt zu erkunden.
Klicken Sie auf eine der Schaltflächen, um Substitutionen, fehlende Zeichen, zusätzliche Zeichen oder Fraktur-Verwechslungen hervorzuheben. Die entsprechenden Stellen im Ground Truth werden automatisch markiert, sodass Sie genau sehen können, wie und wo die OCR vom Originaltext abgewichen ist.
Sie können auch „Fehlende GT anzeigen“ aktivieren, um Zeichen anzuzeigen, die von der OCR übersprungen wurden.

In [ ]:
#@title Interactive OCR error widget (click to expand code)
from IPython.display import display, HTML
display(HTML(r"""
<div id="ocr-error-v4" style="font-family: system-ui, monospace; max-width:1000px; margin: 8px 0;">

  <h3 style="margin:0 0 8px 0; color: inherit;">OCR-Ausgabe (annotiert)</h3>

  <div style="margin:8px 0 12px 0;">
    <button class="ocrBtn" onclick="toggleErrorsV4('sub')">Substitutionen</button>
    <button class="ocrBtn" onclick="toggleErrorsV4('miss')">Fehlende Zeichen (in OCR)</button>
    <button class="ocrBtn" onclick="toggleGTunderV4()">Fehlende GT anzeigen</button>
    <button class="ocrBtn" onclick="toggleErrorsV4('ins')">Zusätzliche Zeichen (in OCR)</button>
    <button class="ocrBtn" onclick="toggleErrorsV4('frak')">Fraktur-Verwechslungen</button>
    <button class="ocrBtn" onclick="clearErrorsV4()">Zurücksetzen</button>
  </div>

  <div id="ocrBoxV4"
       style="white-space:pre-wrap; padding:10px; border-radius:6px;
              background:#1f1f1f; color:#eee; border:1px solid #2f2f2f;
              font-family:monospace;"></div>

  <div style="margin-top:10px; font-size:1.0rem; color: inherit;">
    <strong>Legende:</strong>
    <span style="margin-left:8px; color:#ff6b6b;">Substitution</span>
    <span style="margin-left:8px; color:#47b5ff;">fehlend</span>
    <span style="margin-left:8px; color:#7effa2;">zusätzlich</span>
    <span style="margin-left:8px; color:#ffd54d;">Fraktur</span>
  </div>


  <h3 style="margin:0 0 8px 0; color: inherit;">Ground Truth (Referenz)</h3>
  <div id="gtBoxV4"
       style="white-space:pre-wrap; padding:10px; border-radius:6px;
              background:#29313d; color:#e5e5e5; border:1px solid #2f2f2f;
              font-family:monospace;"></div>

</div>

<style>

  #ocr-error-v4 .ocr-heading {
  margin: 0 0 8px 0 !important;
  color: #222 !important;
  }

  html[data-theme="dark"] #ocr-error-v4 .ocr-heading,
  body[data-theme="dark"] #ocr-error-v4 .ocr-heading {
    color: #eee !important;
  }
  
  #ocr-error-v4 .ocrBtn {
    background:#2d3b4f; color:#fff; border:none;
    padding:6px 10px; margin-right:6px;
    border-radius:5px; cursor:pointer; font-size:0.9rem;
  }

  #ocr-error-v4 .ocrSpan { padding:0 2px; border-radius:3px; display:inline-block; }
  #ocr-error-v4 .ok   { color:#cfcfcf; }
  #ocr-error-v4 .sub  { background:#ff6b6b; color:#000; }
  #ocr-error-v4 .ins  { background:#7effa2; color:#000; }
  #ocr-error-v4 .miss { background:#47b5ff; color:#000; position:relative; }
  #ocr-error-v4 .frak { background:#ffd54d; color:#000; }

  #ocr-error-v4 .gtSpan { transition:color 0.2s ease; }

  #ocr-error-v4 .gt-under {
    display:block; font-size:0.8rem; text-align:center;
    line-height:0.8; opacity:0; transition:opacity 0.25s ease;
    color:#000;
  }
  #ocr-error-v4 .miss.showGT .gt-under { opacity:1; }
</style>

<script>
(function(){
  const ocrRaw = `7ie Grippe wüfel weiter Zunahme der fchweren Fälle in Berlin. Die Zahl der Grippefälle iſt in den leßten be:der Tagen auc<h in Groß-Berlin noH erf>lih zeftiegen. Die Worenhäuſer und ſon- haen aroßen GeſhHöäfte, die Krirgs- unh die prie n Betriebe lagen, daß übermäig viele An- "Fz hcben krer? melden müſſen,-und an< ; .*e* Vofſt und 5ei der Straßenbahn iſt der ſo3 der Grippelrantken bedeuter) g&`.trim();
  const gtRaw = `Die Grippe wütet weiter Zunahme der schweren Fälle in Berlin. Die Zahl der Grippefälle ist in den letzten beiden Tagen auch in Groß-Berlin noch erheblich gestiegen. Die Warenhäuser und sonstigen großen Geschäfte, die Kriegs- und die privaten Betriebe klagen, daß übermäßig viele An-
gestellte sich haben krank melden müssen und auch bei der Post und bei der Straßenbahn ist der Prozentsatz der Grippekranken bedeutend gestiegen.`.trim();

  const ocr = ocrRaw.replace(/\s+/g,' ');
  const gt  = gtRaw.replace(/\s+/g,' ');

  const gtBox = document.getElementById('gtBoxV4');
  gtBox.innerHTML = gt.split('').map(ch=>`<span class="gtSpan">${ch}</span>`).join('');

  /* Levenshtein diff */
  function computeEdits(a,b){
    const n=a.length, m=b.length;
    const dp=Array.from({length:n+1},()=>Array(m+1).fill(0));
    for(let i=0;i<=n;i++) dp[i][0]=i;
    for(let j=0;j<=m;j++) dp[0][j]=j;

    for(let i=1;i<=n;i++){
      for(let j=1;j<=m;j++){
        dp[i][j]=Math.min(
          dp[i-1][j]+1,
          dp[i][j-1]+1,
          dp[i-1][j-1] + (a[i-1]===b[j-1]?0:1)
        );
      }
    }

    const ops=[];
    let i=n,j=m;
    while(i>0||j>0){
      if(i>0&&j>0&&a[i-1]===b[j-1]&&dp[i][j]===dp[i-1][j-1]){
        ops.push({type:'ok', a:a[i-1], b:b[j-1], gi:j-1});
        i--;j--;continue;
      }
      if(i>0&&j>0&&dp[i][j]===dp[i-1][j-1]+1){
        ops.push({type:'sub', a:a[i-1], b:b[j-1], gi:j-1});
        i--;j--;continue;
      }
      if(i>0&&dp[i][j]===dp[i-1][j]+1){
        ops.push({type:'ins', a:a[i-1], b:null, gi:null});
        i--;continue;
      }
      if(j>0&&dp[i][j]===dp[i][j-1]+1){
        ops.push({type:'miss', a:null, b:b[j-1], gi:j-1});
        j--;continue;
      }
    }
    return ops.reverse();
  }

  const edits = computeEdits(ocr,gt);
  const frakPairs = [['ſ','s'],['ſ','f'],['u','n'],['0','o'],['5','s']];
  function isFrak(op){
    if(op.type!=='sub') return false;
    return frakPairs.some(([x,y]) => (op.a===x&&op.b===y)||(op.a===y&&op.b===x));
  }

  const ocrBox = document.getElementById('ocrBoxV4');
  let showGT = false;
  let currentFilter = null;

  function render(){
    ocrBox.innerHTML='';

    edits.forEach(op=>{
      if(op.type==='ok'){
        const s=document.createElement('span');
        s.className='ocrSpan ok';
        s.textContent=op.a;
        s.dataset.type='ok';
        ocrBox.appendChild(s);
      }
      else if(op.type==='sub'){
        const s=document.createElement('span');
        s.className='ocrSpan sub';
        if(isFrak(op)) s.classList.add('frak');
        s.textContent=op.a;
        s.dataset.type='sub';
        s.dataset.gi=op.gi;
        ocrBox.appendChild(s);
      }
      else if(op.type==='ins'){
        const s=document.createElement('span');
        s.className='ocrSpan ins';
        s.textContent=op.a;
        s.dataset.type='ins';
        ocrBox.appendChild(s);
      }
      else if(op.type==='miss'){
        const w=document.createElement('span');
        w.className='ocrSpan miss';
        w.textContent='⟦ ⟧';
        w.dataset.type='miss';
        w.dataset.gi=op.gi;

        const gtline=document.createElement('span');
        gtline.className='gt-under';
        gtline.textContent=op.b;
        w.appendChild(gtline);

        if(showGT) w.classList.add('showGT');
        ocrBox.appendChild(w);
      }
    });

    applyFilter();
  }

  function applyFilter(){
    const ocrSpans = ocrBox.querySelectorAll('.ocrSpan');
    const gtSpans = gtBox.querySelectorAll('.gtSpan');

    gtSpans.forEach(s=>{
      s.style.color='#e5e5e5';
      s.style.fontWeight='normal';
    });

    ocrSpans.forEach(s=>{
      s.style.opacity='1';
      s.style.outline='none';
      s.style.background = '';
      s.style.color = '';
    });

    if(!currentFilter) return;

    ocrSpans.forEach((s,i)=>{
      const t = s.dataset.type;
    
      let match = false;
    
      if(currentFilter === 'sub'){
        match = (t === 'sub' && !isFrak(edits[i]));
      }
      else if(currentFilter === 'frak'){
        match = isFrak(edits[i]);
      }
      else{
        match = (t === currentFilter);
      }
    
      if(!match){
        s.style.opacity='0.45';
      } else {
        s.style.outline='2px solid rgba(255,255,255,0.25)';
        if(currentFilter === 'frak' && isFrak(edits[i])){
          s.style.background = '#ffd54d';
          s.style.color = '#000';
        }
      }
    });
    
    // GT coloring
    edits.forEach(op=>{
      if(op.gi===null) return;
      const gtSpan = gtSpans[op.gi];
      if(!gtSpan) return;

      if(currentFilter==='sub' && op.type==='sub'&& !isFrak(op)){
        gtSpan.style.color='#ff6b6b';
        gtSpan.style.fontWeight='bold';
      }
      if(currentFilter==='miss' && op.type==='miss'){
        gtSpan.style.color='#47b5ff';
        gtSpan.style.fontWeight='bold';
      }
      if(currentFilter==='frak' && isFrak(op)){
        gtSpan.style.color='#ffd54d';
        gtSpan.style.fontWeight='bold';
      }
    });
  }

  window.toggleErrorsV4=function(type){
    currentFilter=type;
    render();
  };

  window.clearErrorsV4=function(){
    currentFilter=null;
    render();
  };

  window.toggleGTunderV4=function(){
    showGT=!showGT;
    render();
  };

  render();  
})();
</script>
"""))

## Verarbeitung eines (mehrseitigen) PDFs

Mit ein wenig mehr Python-Code können wir pytesseract auch verwenden, um gesamte **PDF-Dateien mit vielen Seiten** zu OCRen: <!-- With a bit more Python code, we can also use pytesseract to OCR entire **PDF files with many pages**: -->

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Zuerst wird der Ordner angelegt, in dem die Textdateien gespeichert werden. Der Einfachheit halber wird die gleiche Datenablagestruktur wie in dem <a href="https://github.com/quadriga-dk/Text-Fallstudie-1/tree/main">GitHub Repository</a>, in dem die Daten gespeichert sind, vorausgesetzt. </br>
Der Text wird aus GitHub heruntergeladen und in dem Ordner <i>../data/pdf/</i> abgespeichert. </br>
Der Pfad kann in der Variable <i>sample_pdf_path</i> angepasst werden. Die einzulesenden Daten müssen die Endung `.pdf` haben. </br>
</details>

<details>
  <summary><b>Was ist ein Dateipfad?</b> (klicken)</summary>

Ein **Dateipfad** ist eine Zeichenkette, die deinem Programm sagt, wo eine Datei auf deinem Computer oder Server gespeichert ist. Er hilft dem Programm, Dateien zu finden und auf sie zuzugreifen, um sie zu lesen, zu schreiben oder zu bearbeiten.

#### Arten von Dateipfaden:
1. **Absoluter Dateipfad**:  
   Ein absoluter Pfad gibt den vollständigen Speicherort einer Datei ausgehend vom Stammverzeichnis deines Systems an.
   - Beispiel unter Windows:  
     `C:\Users\JohnDoe\Documents\file.txt`
   - Beispiel unter macOS/Linux:  
     `/Users/JohnDoe/Documents/file.txt`
     
2. **Relativer Dateipfad**:  
   Ein relativer Pfad zeigt dem Programm, wie es eine Datei basierend auf dem aktuellen Arbeitsverzeichnis (dem Ordner, in dem dein Skript ausgeführt wird) finden kann.
   - Beispiel:  
     `Documents/file.txt`  
     (Dies sucht die Datei in einem Ordner namens `Documents` innerhalb des aktuellen Verzeichnisses).

#### Pfadtrennzeichen:
- Unter Windows verwenden Pfade Backslashes (`\`):  
  `C:\folder\file.txt`
- Unter macOS/Linux verwenden Pfade Schrägstriche (`/`):  
  `/folder/file.txt`

#### Beispiel in Python:

```
# Absoluter Pfad
file = open('C:/Users/JohnDoe/Documents/file.txt')
    
# Relativer Pfad
file = open('Documents/file.txt')
```

Python bietet auch Tools, um Pfade so zu handhaben, dass sie auf jedem Betriebssystem funktionieren, wie die Module `os` und `pathlib`. Wir verwenden oben `pathlib`, damit dieses Notebook auf jedem Rechner funktioniert. Dadurch können wir Pfade im Unix-Stil schreiben.
    
</details>

In [ ]:
# 🚀 Create data directory path
corpus_dir = Path("../data/pdf")
if not corpus_dir.exists():
    corpus_dir.mkdir(parents=True)

In [ ]:
# 🚀 Load the pdf file from GitHub 
download_file("https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/data/pdf/SNP27112366-19181224-0-0-0-0.pdf", "../data/pdf")

In [ ]:
# set the path to file to be processed
sample_pdf_path = Path("../data/pdf/SNP27112366-19181224-0-0-0-0.pdf")

Dieser Code liest eine mehrseitige PDF-Datei mit einer Zeitungsausgabe vollständig ein und führt Seite für Seite eine Texterkennung (OCR) durch. Die Ausführung wird mehrere Minuten dauern

In [ ]:
# this code here reads an entire PDF with a newspaper issue 
# and performs OCR page by page
# it will take a couple of minutes to run
recognized_pages = []
converted_pdf = tqdm(convert_from_path(sample_pdf_path, use_cropbox=True))
for image in converted_pdf:
    recognized = pytesseract.image_to_string(image, 
                                             lang='deu_latf') 
    #print(recognized)
    recognized_pages.append(recognized)

Schauen wir uns die erste Seite an: <!-- Let's look at the first page:-->

In [ ]:
print(recognized_pages[0])

Letzte Seite: <!-- Last page: --> 

In [ ]:
print(recognized_pages[-1])

Keines dieser Ergebnisse sieht besonders gut aus (hauptsächlich aufgrund der Scan-Qualität und allgemeiner Herausforderungen bei der Arbeit mit alten Zeitungen). In den nächsten Abschnitten werden wir lernen, wie man
* a) die OCR-Qualität misst
* b) die Qualität in der OCR-Nachkorrekturphase verbessert

<!-- None of these results look very good (mostly due to scan quality and general challenges of working with old newspapers). In the next parts we will learn how to 
* a) measure the OCR quality
* b) improve the quality at the OCR postcorrection stage -->

Um die OCR-Funktion auf einer anderen PDF-Datei auszuführen, müssen Sie in der obigen Zeile einen Dateipfad dazu angeben: `sample_pdf_path = Path('/path/to/your.pdf')`. 

## (Advanced) Verarbeitung des gesamten Korpus von PDFs mit derselben OCR-Engine <!-- (Advanced) Processing the whole corpus of PDF-s with the same OCR engine -->

Der untenstehende Code verarbeitet alle Dateien im Ordner `'../data/pdf'`, die die Endung '.pdf' haben, und speichert die Ergebnisse dann im Ordner `'../data/txt'` (die Dateinamen bleiben gleich, aber mit der Endung '.txt' anstelle von '.pdf'). **WARNUNG**: Bei einer großen Anzahl (>5) von PDFs wird dies viel Zeit in Anspruch nehmen.

<!-- The code below will process all the files in folder `'../data/pdf'` which have '.pdf' as extension, and then put the results into the `'../data/txt'` (the filenames will be the same but with '.txt' extension instead of '.pdf'). **WARNING**: For a large (>5) number of PDF-s this will take a long time. -->

In [ ]:
# 🚀 Create txt directory path
corpus_dir = Path("../data/txt")
if not corpus_dir.exists():
    corpus_dir.mkdir(parents=True)

In [ ]:
pathpdf = Path('../data/pdf')
pathtxt = Path('../data/txt')

In [ ]:
for filename in tqdm(pathpdf.iterdir()):
    if filename.suffix == '.pdf':
        converted_pdf = convert_from_path(filename, use_cropbox=True)
        output_path = pathtxt / filename.stem 
        output_path = output_path.with_suffix('.txt')
        with output_path.open('w', encoding='utf-8') as output_txt:
            for image in converted_pdf:
                recognized = pytesseract.image_to_string(image, 
                                                         lang='deu_latf') 
                output_txt.write(recognized)